# Capstone: Build a mini-GPT from scratch

_Capstones_

**Seven landmark papers, built in order, become one small character-level GPT that generates text.**

---

This notebook is a practice scaffold for the **Capstone: Build a mini-GPT from scratch** lesson. We assemble the model one labelled piece at a time — the character corpus, causal self-attention, the pre-norm block, the full nanoGPT, and the AdamW training loop — and watch the generated text go from gibberish to readable.

It is deliberately broken into small steps, and after each one we **stop to look at the data**: the vocabulary as a table, the causal mask as a heatmap, attention weights, a parameter count, the loss curve, and the next-character probabilities. Run each cell top to bottom and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

Before building anything, see exactly what the model will read. A language model just predicts the **next character**, so our "data" is a string, a table that maps each character to an integer id, and batches of `(input, next-char)` pairs. We build and inspect all three here.

### Step 1 — The character corpus

We train on a tiny, repeated snippet of text so the run is fast and the model can actually memorise the pattern. Working at the **character** level keeps the vocabulary tiny (just the distinct characters), so the whole pipeline stays small enough to read end to end.

In [ ]:
# A tiny char-level corpus: two short lines, repeated so there is enough to learn from.
text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50

print("total characters :", len(text))
print("distinct characters:", len(set(text)))
print("first 90 characters:")
print(repr(text[:90]))

### Step 2 — The vocabulary (every character gets an id)

A neural network consumes numbers, not letters, so we assign each distinct character an integer id. `stoi` ("string-to-int") and `itos` ("int-to-string") are the two lookup directions. The vocabulary size `VOCAB` matters: a freshly initialised model guesses uniformly over `VOCAB` characters, so its loss should start near `ln(VOCAB)`.

The table below lists every character with its id and how often it appears; the bar chart shows the same frequencies (spaces and common letters dominate).

In [ ]:
import collections

chars = sorted(set(text))
stoi  = {c: i for i, c in enumerate(chars)}      # char -> id
itos  = {i: c for c, i in stoi.items()}          # id   -> char
VOCAB = len(chars)
counts = collections.Counter(text)

vocab_df = pd.DataFrame({
    "id":    [stoi[c] for c in chars],
    "char":  [repr(c) for c in chars],
    "count": [counts[c] for c in chars],
})
print("VOCAB size:", VOCAB,
      " ->  random-init loss should start near ln(VOCAB) =", round(float(np.log(VOCAB)), 3))
display(vocab_df)

plt.figure(figsize=(9, 2.6))
plt.bar([repr(c) for c in chars], [counts[c] for c in chars], color="#4ea1ff")
plt.title("character frequencies in the corpus")
plt.ylabel("count")
plt.show()

### Step 3 — Tokenize: turn text into integer ids

"Tokenizing" here just means mapping each character through `stoi`. The little table shows one phrase encoded character by character; then we encode the **whole corpus** into one long 1-D tensor `data` — the stream the model will slide a window over.

In [ ]:
import torch

def encode(s):
    return [stoi[c] for c in s]

phrase = "to be"
enc_df = pd.DataFrame({
    "position": range(len(phrase)),
    "char":     [repr(c) for c in phrase],
    "id":       encode(phrase),
})
display(enc_df)

data = torch.tensor(encode(text))
print("encoded corpus tensor:", tuple(data.shape), data.dtype)
print("first 20 ids:", data[:20].tolist())

### Step 4 — Batches and the next-character target (shift by one)

For every position the **label is simply the next character**, so the target `y` is the input `x` shifted left by one. `get_batch` draws `B` random windows of length `SEQ` from the corpus. The table aligns the first window's input characters with their targets — read it as "after `'t'` comes `'o'`, after `'o'` comes `' '`, …".

In [ ]:
SEQ, B = 32, 64                  # window length, batch size
torch.manual_seed(0)

def get_batch():
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ]     for i in ix])   # the window
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])  # the SAME window shifted left by 1
    return x, y

xb, yb = get_batch()
print("x batch:", tuple(xb.shape), "  y batch:", tuple(yb.shape))

k = 12
shift_df = pd.DataFrame({
    "pos":            range(k),
    "input id":       xb[0, :k].tolist(),
    "input char":     [repr(itos[i]) for i in xb[0, :k].tolist()],
    "-> target char": [repr(itos[i]) for i in yb[0, :k].tolist()],
    "target id":      yb[0, :k].tolist(),
})
display(shift_df)   # each input char's label is just the NEXT char

## Reference implementation — PyTorch

The finished model is a **decoder-only Transformer** (the GPT family), and every piece traces to one landmark paper. Here is the build plan we are about to follow:

| Paper | Contributes | In our code |
|---|---|---|
| word2vec | token embedding table | `nn.Embedding(VOCAB, d_model)` |
| attention | scaled dot-product attention | `Q @ Kᵀ / √dₖ` then `softmax` |
| transformer | multi-head attention, positions, the block | `CausalSelfAttention`, `nn.Embedding(max_len, …)`, `Block` |
| layernorm | pre-norm normalization | `nn.LayerNorm` before each sublayer |
| resnet | residual connections | the two `x + sublayer(x)` lines |
| adamw | decoupled weight decay | `torch.optim.AdamW` |
| gpt | causal mask + next-token loss + sampling | `masked_fill(mask, -inf)`, `cross_entropy`, `generate` |

### Step 5 — Causal multi-head self-attention

Self-attention lets each position mix in information from other positions. We project the input into queries, keys and values, split them into `h` heads, and score every pair with the scaled dot product `Q·Kᵀ / √d_k` (the *attention* paper). A softmax turns scores into weights, which we use to average the values.

The **causal mask** is GPT's twist: we set the scores *above* the diagonal to `-inf` so a position can never attend to future tokens — essential for predicting the next character from only the ones before it. Finally the heads are concatenated and mixed by `W_o`.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

# Causal multi-head self-attention (paper-attention + paper-transformer + paper-gpt's mask).
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, h, max_len):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # paper-gpt's causal mask: True ABOVE the diagonal = the future, which we forbid.
        self.register_buffer("mask", torch.triu(torch.ones(max_len, max_len), diagonal=1).bool())

    def _split(self, x):                                    # (B,S,d_model) -> (B,h,S,d_k): the "heads"
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self._split(self.W_q(x))
        K = self._split(self.W_k(x))
        V = self._split(self.W_v(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)         # scaled dot product (paper-attention)
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))  # forbid the future (paper-gpt)
        weights = F.softmax(scores, dim=-1)
        out = weights @ V                                              # attention weights @ values
        out = out.transpose(1, 2).contiguous().view(B, S, self.h * self.d_k)  # concat the heads
        return self.W_o(out)

### Step 6 — See the causal mask (no peeking at the future)

The mask is the whole reason this is a *language* model: query position `i` may only attend to key positions `j ≤ i`. Plotting it makes the rule literal — the upper triangle (the future) is blocked, the lower triangle (the past and present) is allowed.

In [ ]:
L = 8
mask = torch.triu(torch.ones(L, L), diagonal=1).bool()   # True above the diagonal = forbidden future

plt.figure(figsize=(5.4, 4))
plt.imshow((~mask).int(), cmap="Blues")                  # 1 = allowed, 0 = blocked
plt.title("causal mask: query i attends to keys j <= i")
plt.xlabel("key position j"); plt.ylabel("query position i")
plt.xticks(range(L)); plt.yticks(range(L))
plt.colorbar(label="allowed (1) / blocked (0)")
plt.show()

### Step 7 — Watch one attention layer run

Let us push a small random input through a fresh attention layer and read the **weights** it produces. Two things to notice: the output keeps the input's `(batch, seq, d_model)` shape, and every row of the weight matrix is **lower-triangular and sums to 1** — exactly the causal rule from Step 6, now enforced by the softmax over masked scores.

In [ ]:
torch.manual_seed(0)
demo_attn = CausalSelfAttention(d_model=64, h=4, max_len=SEQ)
S = 10
x_demo = torch.randn(1, S, 64)

with torch.no_grad():
    # recompute the weights the layer uses internally, for one head, so we can see them
    Q = demo_attn._split(demo_attn.W_q(x_demo))
    K = demo_attn._split(demo_attn.W_k(x_demo))
    scores = Q @ K.transpose(-2, -1) / math.sqrt(demo_attn.d_k)
    scores = scores.masked_fill(demo_attn.mask[:S, :S], float("-inf"))
    w = torch.softmax(scores, dim=-1)            # (1, h, S, S)
    out = demo_attn(x_demo)

print("input :", tuple(x_demo.shape), "  output:", tuple(out.shape), "  weights:", tuple(w.shape))
print("row sums, head 0:", [round(v, 3) for v in w[0, 0].sum(-1).tolist()])  # each row -> 1

plt.figure(figsize=(4.6, 4))
plt.imshow(w[0, 0], cmap="viridis")
plt.title("attention weights — head 0 (lower-triangular)")
plt.xlabel("key position"); plt.ylabel("query position")
plt.colorbar(label="weight")
plt.show()

### Step 8 — The pre-norm transformer block

A `Block` wraps two sublayers: the causal attention from Step 5 and a small feed-forward MLP. Each sublayer is *pre-normed* — a `LayerNorm` is applied to the input before the sublayer (the LayerNorm paper).

The two `x + ...` lines are ResNet's residual connections: the block adds its output back onto the input rather than replacing it, which keeps gradients flowing through deep stacks. Because of those adds, a block's output has exactly the same shape as its input — the "residual stream" — which we confirm below.

In [ ]:
# Pre-norm block: LayerNorm -> sublayer -> residual add (paper-layernorm + paper-resnet).
class Block(nn.Module):
    def __init__(self, d_model, h, max_len, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)                    # paper-layernorm (pre-norm: LN at the input)
        self.attn = CausalSelfAttention(d_model, h, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # residual around masked attention (paper-resnet)
        x = x + self.ff(self.ln2(x))       # residual around feed-forward    (paper-resnet)
        return x

# A block maps (B, S, d_model) -> the SAME shape (the residual stream is preserved).
demo_block = Block(d_model=64, h=4, max_len=SEQ, d_ff=128)
probe = torch.randn(1, S, 64)
print("block in :", tuple(probe.shape))
print("block out:", tuple(demo_block(probe).shape), " (unchanged)")

### Step 9 — Assemble the nanoGPT

The full model adds a **token embedding** (word2vec's lookup table) and a **learned positional embedding** (so the model knows token order), stacks `n_layers` blocks, applies a final `LayerNorm`, and projects to per-character logits with the language-model `head`.

`generate` runs the model autoregressively: each step it takes the logits at the last position, divides by a temperature, softmaxes to probabilities, and *samples* the next character — GPT's sampling loop — appending it and repeating.

In [ ]:
# The nanoGPT: token + learned positional embeddings, N blocks, final LN, vocab head.
class NanoGPT(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, n_layers=3, max_len=64, d_ff=128):
        super().__init__()
        self.max_len = max_len                              # longest context the position table can index
        self.tok = nn.Embedding(vocab, d_model)             # token embedding table (paper-word2vec)
        self.pos = nn.Embedding(max_len, d_model)           # LEARNED positional embedding (paper-transformer)
        self.blocks = nn.ModuleList([Block(d_model, h, max_len, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)                   # GPT-2's extra final LayerNorm
        self.head = nn.Linear(d_model, vocab)               # the language-model head -> per-char logits

    def forward(self, idx):
        B, S = idx.shape
        pos = torch.arange(S, device=idx.device)
        x = self.tok(idx) + self.pos(pos)                   # add token + positional embeddings
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))                      # (B,S,vocab) next-char logits

    @torch.no_grad()
    def generate(self, idx, n_new, temp=0.8):
        for _ in range(n_new):
            logits = self(idx[:, -self.max_len:])[:, -1, :] / temp   # logits for the next char at the last slot
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)               # SAMPLE the next char (paper-gpt)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### Step 10 — Count the parameters and check the random-init loss

Two quick sanity checks before training. First, tabulate the trainable parameters by component — a good way to see where the model's capacity lives (the embeddings and the per-block attention/MLP weights). Second, run one **untrained** batch through the model: its cross-entropy should land right around `ln(VOCAB)`, because an untrained model has no idea which character comes next and spreads its bets evenly.

In [ ]:
torch.manual_seed(0)
net = NanoGPT(VOCAB, max_len=SEQ)

# Group parameters by component so the budget is readable at a glance.
import re
group_counts = {}
for name, p in net.named_parameters():
    m = re.match(r"(blocks\.\d+)", name)
    comp = m.group(1) if m else name.split(".")[0]
    group_counts[comp] = group_counts.get(comp, 0) + p.numel()

total = sum(group_counts.values())
param_df = pd.DataFrame({
    "component":  list(group_counts),
    "parameters": list(group_counts.values()),
    "% of total": [round(100 * v / total, 1) for v in group_counts.values()],
})
display(param_df)
print("total trainable parameters:", f"{total:,}")

with torch.no_grad():
    xb, yb = get_batch()
    loss0 = F.cross_entropy(net(xb).reshape(-1, VOCAB), yb.reshape(-1)).item()
print(f"untrained loss = {loss0:.3f}   vs   ln(VOCAB) = {float(np.log(VOCAB)):.3f}   (should match)")

### Step 11 — Train with AdamW (recording the loss curve)

We optimise the model with **AdamW** (Adam with decoupled weight decay) on the next-character cross-entropy loss — GPT's causal-LM objective. Each step we forward a batch, compute the loss over every position, and backprop.

This time we **record** the loss at every step and a generated sample every 500 steps, so the next section can plot how training actually went. The loss should fall from ~3.0 toward ~0.2 and the samples should drift from gibberish to readable text. (Our small run; exact values vary by hardware and seed.)

In [ ]:
def sample(net, n_new=60, seed_char="t"):
    start = torch.tensor([[stoi[seed_char]]])
    out = net.generate(start, n_new)[0].tolist()
    return "".join(itos[i] for i in out)

opt = torch.optim.AdamW(net.parameters(), lr=3e-3)         # paper-adamw: decoupled weight decay
loss_hist, sample_log = [], []
for step in range(2001):
    x, y = get_batch()
    logits = net(x)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))  # causal-LM objective (paper-gpt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    loss_hist.append((step, loss.item()))
    if step % 500 == 0:
        s = sample(net)
        sample_log.append((step, s))
        print(f"step {step:4d}  loss {loss.item():.3f}   sample: {s!r}")
# Loss falls from ~3.0 (random over ~20 chars) toward ~0.2, and samples go gibberish -> readable.
# Our small run, not a paper's reported number; exact values vary by hardware and seed.

## Visualize the data & results

_As the assembled char-level mini-GPT trains with AdamW on the next-character cross-entropy objective, does the loss fall and does the generated text become readable?_ We answer with the history we just recorded — no retraining needed.

### Step 12 — The loss curve

Plotting every recorded step shows the full optimisation trajectory. The dashed line marks `ln(VOCAB)`, the loss of a model that has learned nothing; a working build starts there and drops well below it.

In [ ]:
steps  = [s for s, _ in loss_hist]
losses = [l for _, l in loss_hist]

plt.figure(figsize=(7.5, 3.6))
plt.plot(steps, losses, color="#7ee787", lw=1.2, label="train loss")
plt.axhline(float(np.log(VOCAB)), color="#ff6b6b", ls="--",
            label=f"ln(VOCAB) = {float(np.log(VOCAB)):.2f}  (random guessing)")
plt.xlabel("training step"); plt.ylabel("next-char cross-entropy (nats)")
plt.title("loss falls from ~ln(VOCAB) toward ~0")
plt.legend(); plt.show()

print("first-step loss:", round(losses[0], 3), "   final-step loss:", round(losses[-1], 3))

### Step 13 — Samples: gibberish → readable

The same story in words. Each row is a 60-character sample (seeded with `'t'`) taken at that training step. Early rows are noise; later rows reproduce fragments of the corpus.

In [ ]:
sample_df = pd.DataFrame(sample_log, columns=["step", "generated sample (60 chars, seeded with 't')"])
display(sample_df)

### Step 14 — What the trained model predicts next

A language model outputs a probability distribution over the next character. We feed it the context `"to be, or not to b"` and bar-chart its top predictions for the next character — it should put almost all of its mass on `'e'` (…to b → e).

In [ ]:
context = "to be, or not to b"
ids = torch.tensor([encode(context)])
with torch.no_grad():
    logits = net(ids)[:, -1, :]            # logits at the last position
    probs = F.softmax(logits, dim=-1)[0]

top = torch.argsort(probs, descending=True)[:10]
plt.figure(figsize=(7, 3))
plt.bar([repr(itos[i.item()]) for i in top], probs[top].tolist(), color="#79c0ff")
plt.title(f"P(next character | {context!r})")
plt.ylabel("probability")
plt.show()

print("context           :", repr(context))
print("top next character:", repr(itos[top[0].item()]), " with p =", round(probs[top[0]].item(), 3))

### Step 15 — The trained model's attention

Finally, read the attention weights of the **trained** model on that same context (block 0, head 0). The axes are the actual characters; the lower-triangular pattern is the causal mask, and the bright cells show which earlier characters each position leans on.

In [ ]:
ids = torch.tensor([encode(context)])
S = ids.shape[1]
a0 = net.blocks[0].attn
with torch.no_grad():
    x_emb = net.tok(ids) + net.pos(torch.arange(S))
    x_ln = net.blocks[0].ln1(x_emb)                       # pre-norm input the attention actually sees
    Q = a0._split(a0.W_q(x_ln))
    K = a0._split(a0.W_k(x_ln))
    sc = Q @ K.transpose(-2, -1) / math.sqrt(a0.d_k)
    sc = sc.masked_fill(a0.mask[:S, :S], float("-inf"))
    w = torch.softmax(sc, dim=-1)[0, 0]                   # head 0

fig, ax = plt.subplots(figsize=(6, 5.2))
im = ax.imshow(w, cmap="viridis")
ax.set_xticks(range(S)); ax.set_xticklabels(list(context))
ax.set_yticks(range(S)); ax.set_yticklabels(list(context))
ax.set_xlabel("attends to (key)"); ax.set_ylabel("query position")
ax.set_title("trained attention — block 0, head 0")
fig.colorbar(im, label="weight")
plt.show()